<a href="https://colab.research.google.com/github/felariop-jpg/macro-nowcasting-dashboard/blob/main/Macro_Nowcasting_Dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [45]:
# === Cell 1: Environment setup ===========================================
# Install the few libraries Colab does not already ship. Quiet to keep logs clean.
!pip -q install fredapi plotly kaleido statsmodels scikit-learn 2>/dev/null

import os, time, warnings, pickle
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
import statsmodels.api as sm
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss, confusion_matrix
from fredapi import Fred

warnings.filterwarnings('ignore')
pio.templates.default = 'plotly_white'
pd.set_option('display.width', 140)

print('Imports OK. pandas', pd.__version__)

Imports OK. pandas 2.2.2


In [46]:
# === Cell 2: Configuration and FRED API key ==============================
from getpass import getpass

# Free key: https://fredaccount.stlouisfed.org/apikeys
FRED_API_KEY = os.environ.get('FRED_API_KEY', '').strip()
if not FRED_API_KEY:
    FRED_API_KEY = getpass('Enter your FRED API key: ').strip()

assert len(FRED_API_KEY) >= 16, 'That FRED key looks too short. Double-check and rerun.'
fred = Fred(api_key=FRED_API_KEY)

START_DATE = '1970-01-01'

# Indicator universe.
#   transform: 'yoy'      -> 12-month percent change   (trending level series)
#              'yoy_diff' -> 12-month change in level   (rate series)
#              'level'    -> use as-is                  (spreads, ratios, indices)
#   sign:      +1 if higher = stronger economy, -1 if higher = weaker economy
SERIES_CONFIG = {
    # Labor
    'PAYEMS':       dict(name='Nonfarm Payrolls',         category='Labor',       transform='yoy',      sign=+1),
    'UNRATE':       dict(name='Unemployment Rate',        category='Labor',       transform='yoy_diff', sign=-1),
    'ICSA':         dict(name='Initial Jobless Claims',   category='Labor',       transform='yoy',      sign=-1),
    'CCSA':         dict(name='Continued Claims',         category='Labor',       transform='yoy',      sign=-1),
    'AWHMAN':       dict(name='Weekly Hours Mfg',         category='Labor',       transform='yoy',      sign=+1),
    'TEMPHELPS':    dict(name='Temp Help Services',       category='Labor',       transform='yoy',      sign=+1),
    'MANEMP':       dict(name='Manufacturing Payrolls',   category='Labor',       transform='yoy',      sign=+1),
    'JTSJOL':       dict(name='Job Openings (JOLTS)',     category='Labor',       transform='yoy',      sign=+1),
    # Production
    'INDPRO':       dict(name='Industrial Production',    category='Production',  transform='yoy',      sign=+1),
    'IPMAN':        dict(name='IP: Manufacturing',        category='Production',  transform='yoy',      sign=+1),
    'TCU':          dict(name='Capacity Utilization',     category='Production',  transform='level',    sign=+1),
    'DGORDER':      dict(name='Durable Goods Orders',     category='Production',  transform='yoy',      sign=+1),
    'NEWORDER':     dict(name='Core Capital Goods Orders',category='Production',  transform='yoy',      sign=+1),
    # Consumption and income
    'RSAFS':        dict(name='Retail Sales',             category='Consumption', transform='yoy',      sign=+1),
    'PCEC96':       dict(name='Real PCE',                 category='Consumption', transform='yoy',      sign=+1),
    'DSPIC96':      dict(name='Real Disposable Income',   category='Consumption', transform='yoy',      sign=+1),
    'W875RX1':      dict(name='Real Income ex Transfers', category='Consumption', transform='yoy',      sign=+1),
    'CMRMTSPL':     dict(name='Real Mfg+Trade Sales',     category='Consumption', transform='yoy',      sign=+1),
    'UMCSENT':      dict(name='Consumer Sentiment',       category='Consumption', transform='level',    sign=+1),
    # Housing
    'HOUST':        dict(name='Housing Starts',           category='Housing',     transform='yoy',      sign=+1),
    'PERMIT':       dict(name='Building Permits',         category='Housing',     transform='yoy',      sign=+1),
    'HSN1F':        dict(name='New Home Sales',           category='Housing',     transform='yoy',      sign=+1),
    'TTLCONS':      dict(name='Construction Spending',    category='Housing',     transform='yoy',      sign=+1),
    # Financial conditions
    'T10Y3M':       dict(name='Yield Curve 10Y-3M',       category='Financial',   transform='level',    sign=+1),
    'T10Y2Y':       dict(name='Yield Curve 10Y-2Y',       category='Financial',   transform='level',    sign=+1),
    'BAA10Y':       dict(name='Baa Credit Spread',        category='Financial',   transform='level',    sign=-1),
    'BAMLH0A0HYM2': dict(name='High Yield OAS',           category='Financial',   transform='level',    sign=-1),
    'NFCI':         dict(name='Financial Conditions',     category='Financial',   transform='level',    sign=-1),
    'STLFSI4':      dict(name='Financial Stress Index',   category='Financial',   transform='level',    sign=-1),
    'VIXCLS':       dict(name='VIX',                      category='Financial',   transform='level',    sign=-1),
}

RECESSION_SERIES = 'USREC'   # NBER recession flag, monthly, 1 = recession
CAT_ORDER = ['Labor', 'Production', 'Consumption', 'Housing', 'Financial']

print('Configured', len(SERIES_CONFIG), 'indicators across', len(CAT_ORDER), 'categories.')

Enter your FRED API key: ··········
Configured 30 indicators across 5 categories.


In [47]:
# === Cell 3: Data collection pipeline ====================================
# Robust FRED fetch: per-series retries, graceful failure, and a disk cache so
# re-runs are instant and reproducible.

CACHE_FILE = 'fred_raw_cache.pkl'

def fetch_series(code, start=START_DATE, retries=3, pause=1.0):
    # Return a clean pandas Series for one FRED code, or None after repeated failure.
    for attempt in range(1, retries + 1):
        try:
            s = fred.get_series(code, observation_start=start).dropna()
            if s.empty:
                print('  [warn] %-12s returned no data' % code); return None
            s.name = code
            return s
        except Exception as e:
            print('  [retry %d/%d] %-12s %s' % (attempt, retries, code, str(e)[:70]))
            time.sleep(pause * attempt)
    print('  [fail] %-12s could not be downloaded' % code)
    return None

def collect_raw(use_cache=True):
    if use_cache and os.path.exists(CACHE_FILE):
        with open(CACHE_FILE, 'rb') as f:
            cache = pickle.load(f)
        print('Loaded', len(cache), 'series from cache (delete', CACHE_FILE, 'to force refresh).')
        return cache

    raw, codes = {}, list(SERIES_CONFIG) + [RECESSION_SERIES]
    for i, c in enumerate(codes, 1):
        print('Fetching %2d/%2d  %s' % (i, len(codes), c))
        s = fetch_series(c)
        if s is not None:
            raw[c] = s

    with open(CACHE_FILE, 'wb') as f:
        pickle.dump(raw, f)
    print('Downloaded', len(raw), 'of', len(codes), 'series.')
    return raw

# Force fresh download on first run
raw = collect_raw(use_cache=False)
available = [c for c in SERIES_CONFIG if c in raw]
missing = [c for c in SERIES_CONFIG if c not in raw]

if missing:
    print('Missing (skipped, pipeline continues):', missing)
assert RECESSION_SERIES in raw, 'Recession flag failed to download; cannot proceed.'
print('Usable indicators:', len(available))

Fetching  1/31  PAYEMS
Fetching  2/31  UNRATE
Fetching  3/31  ICSA
Fetching  4/31  CCSA
Fetching  5/31  AWHMAN
Fetching  6/31  TEMPHELPS
Fetching  7/31  MANEMP
Fetching  8/31  JTSJOL
Fetching  9/31  INDPRO
Fetching 10/31  IPMAN
Fetching 11/31  TCU
Fetching 12/31  DGORDER
Fetching 13/31  NEWORDER
Fetching 14/31  RSAFS
Fetching 15/31  PCEC96
Fetching 16/31  DSPIC96
Fetching 17/31  W875RX1
Fetching 18/31  CMRMTSPL
Fetching 19/31  UMCSENT
Fetching 20/31  HOUST
Fetching 21/31  PERMIT
Fetching 22/31  HSN1F
Fetching 23/31  TTLCONS
Fetching 24/31  T10Y3M
Fetching 25/31  T10Y2Y
Fetching 26/31  BAA10Y
Fetching 27/31  BAMLH0A0HYM2
Fetching 28/31  NFCI
Fetching 29/31  STLFSI4
Fetching 30/31  VIXCLS
Fetching 31/31  USREC
Downloaded 31 of 31 series.
Usable indicators: 30


In [48]:
# === Cell 4: Frequency alignment and stationarity transforms =============

def to_monthly(s):
    # Month-end frequency; average any higher-frequency observations in the month.
    return s.resample('ME').mean()

def apply_transform(s, how):
    if how == 'yoy':       return s.pct_change(12) * 100.0
    if how == 'yoy_diff':  return s.diff(12)
    if how == 'level':     return s
    raise ValueError('unknown transform ' + how)

monthly = {}
for c in available:
    monthly[c] = apply_transform(to_monthly(raw[c]), SERIES_CONFIG[c]['transform'])

features = pd.DataFrame(monthly)
features = features[features.index >= START_DATE]

# NBER recession flag on the same monthly grid.
usrec = to_monthly(raw[RECESSION_SERIES]).reindex(features.index).ffill()
usrec = (usrec > 0.5).astype(int)

print('Feature panel:', features.shape,
      '| range', features.index.min().date(), 'to', features.index.max().date())

Feature panel: (678, 30) | range 1970-01-31 to 2026-06-30


In [49]:
# === Cell 5: Standardization (z-scores) and sign alignment ===============
# Convert each indicator to a full-history z-score, then sign it so that a
# POSITIVE value always means a stronger economy.

def zscore(df):
    return (df - df.mean()) / df.std(ddof=0)

z = zscore(features)
for c in z.columns:
    z[c] = z[c] * SERIES_CONFIG[c]['sign']

# Order columns by category for clean displays.
ordered_codes = sorted(z.columns,
    key=lambda c: (CAT_ORDER.index(SERIES_CONFIG[c]['category']), SERIES_CONFIG[c]['name']))
z = z[ordered_codes]
NAMES = {c: SERIES_CONFIG[c]['name'] for c in ordered_codes}

print('Standardized panel ready:', z.shape)
z[ordered_codes].tail(2).T.round(2)

Standardized panel ready: (678, 30)


,2026-05-31,2026-06-30
CCSA,0.19,0.19
ICSA,0.17,0.13
JTSJOL,NaN,NaN
MANEMP,0.05,NaN
PAYEMS,-0.52,NaN
TEMPHELPS,-0.37,NaN
UNRATE,-0.01,NaN
AWHMAN,0.91,NaN
TCU,-0.83,NaN
NEWORDER,NaN,NaN


In [50]:
# === Cell 6: Composite activity index via PCA ============================
# The first principal component of the standardized panel is the composite
# activity factor. Ragged edges (recent months where slow series have not yet
# printed) are filled with 0, the neutral z-score, so they add no information.

z_filled = z.fillna(0.0)
coverage = z.notna().mean(axis=1)        # fraction of indicators available each month
fit_mask = coverage >= 0.70              # fit PCA where coverage is broad

pca = PCA(n_components=5)
pca.fit(z_filled[fit_mask])
pc1 = pd.Series(pca.transform(z_filled)[:, 0], index=z.index, name='PC1')
loadings = pca.components_[0].copy()

# Orient so higher = stronger (positive correlation with the simple average).
simple = z.mean(axis=1)
if pc1.corr(simple) < 0:
    pc1, loadings = -pc1, -loadings

composite = (pc1 - pc1.mean()) / pc1.std(ddof=0)         # standardized for readability
loadings = pd.Series(loadings, index=z.columns, name='loading')
pca_mean = pd.Series(pca.mean_, index=z.columns)
evr = pca.explained_variance_ratio_

print('PC1 explains %.1f%% of common variance; first 3 PCs %.1f%%.'
      % (evr[0] * 100, evr[:3].sum() * 100))
print('Composite vs simple-average correlation: %.3f' % composite.corr(simple))
print('Latest composite reading: %+.2f std devs from average' % composite.iloc[-1])

PC1 explains 42.6% of common variance; first 3 PCs 65.3%.
Composite vs simple-average correlation: 0.926
Latest composite reading: +0.19 std devs from average


In [51]:
# === Cell 7: Recession probability models ================================
# (a) NOWCAST  (coincident): P(in recession now) from composite + yield slope.
# (b) FORECAST (leading)   : P(recession within ~12 months) from the 10Y-3M spread.

HAS_SLOPE = 'T10Y3M' in features.columns

def fit_probit(X, y):
    # Probit with a graceful fallback to regularized logistic on separation/failure.
    Xc = sm.add_constant(X)
    try:
        m = sm.Probit(y, Xc).fit(disp=0, maxiter=200)
        return 'probit', m, pd.Series(m.predict(Xc), index=X.index)
    except Exception as e:
        print('  probit -> logistic fallback:', str(e)[:60])
        lr = LogisticRegression(C=1.0, max_iter=2000).fit(X.values, y.values)
        return 'logit', lr, pd.Series(lr.predict_proba(X.values)[:, 1], index=X.index)

def apply_model(kind, model, X):
    if kind == 'probit':
        return pd.Series(model.predict(sm.add_constant(X)), index=X.index)
    return pd.Series(model.predict_proba(X.values)[:, 1], index=X.index)

# (a) Nowcast
now_cols = {'composite': composite}
if HAS_SLOPE:
    now_cols['slope'] = features['T10Y3M']

now_df = pd.concat(list(now_cols.values()) + [usrec.rename('rec')], axis=1).dropna()
now_df.columns = list(now_cols.keys()) + ['rec']
now_kind, now_model, now_prob = fit_probit(now_df[list(now_cols.keys())], now_df['rec'])

# (b) 12-month-ahead leading model (yield curve only)
if HAS_SLOPE:
    lead_df = pd.concat([features['T10Y3M'].rename('slope'),
                         usrec.shift(-12).rename('rec_fwd')], axis=1).dropna()
    lead_kind, lead_model, lead_prob_fit = fit_probit(lead_df[['slope']], lead_df['rec_fwd'])
    slope_all = features['T10Y3M'].rename('slope').dropna().to_frame()
    lead_prob = apply_model(lead_kind, lead_model, slope_all)
else:
    lead_kind = None
    lead_prob = pd.Series(dtype=float)
    lead_prob_fit = pd.Series(dtype=float)
    lead_df = pd.DataFrame()

print('Nowcast model:', now_kind, '| predictors:', list(now_cols.keys()))
print('Latest nowcast recession probability  : %.1f%%' % (now_prob.iloc[-1] * 100))
if HAS_SLOPE:
    print('Latest 12m-ahead recession probability: %.1f%%' % (lead_prob.iloc[-1] * 100))

Nowcast model: probit | predictors: ['composite', 'slope']
Latest nowcast recession probability  : 2.7%
Latest 12m-ahead recession probability: 9.3%


In [52]:
# === Cell 8: Visualization helpers =======================================

PALETTE = dict(strong='#1b7837', weak='#b2182b', line='#2166ac',
               lead='#ef8a62', rec='rgba(140,140,140,0.18)')

def recession_spans(rec):
    # Contiguous (start, end) windows where the NBER flag is 1.
    spans, in_rec, start = [], False, None
    for dt, v in rec.items():
        if v == 1 and not in_rec: in_rec, start = True, dt
        elif v == 0 and in_rec:   in_rec = False; spans.append((start, dt))
    if in_rec: spans.append((start, rec.index[-1]))
    return spans

REC_SPANS = recession_spans(usrec)

def add_recession_shading(fig, spans=REC_SPANS, row=None, col=None):
    for s, e in spans:
        fig.add_vrect(x0=s, x1=e, fillcolor=PALETTE['rec'], line_width=0,
                      layer='below', row=row, col=col)
    return fig

print('Helpers ready;', len(REC_SPANS), 'recession episodes detected.')

Helpers ready; 8 recession episodes detected.


In [53]:
# === Cell 9: Macro heatmap of standardized readings ======================
LOOKBACK = 36
hm = z[ordered_codes].tail(LOOKBACK).T
hm.index = [NAMES[c] for c in hm.index]

fig_hm = go.Figure(go.Heatmap(
    z=hm.values, x=[d.strftime('%b %Y') for d in hm.columns], y=hm.index,
    colorscale='RdBu', zmid=0, zmin=-3, zmax=3, colorbar=dict(title='z'),
    hovertemplate='%{y}<br>%{x}<br>z = %{z:.2f}<extra></extra>'))

fig_hm.update_layout(title='Macro Indicators: Standardized Readings (last %d months)' % LOOKBACK,
                     height=760, xaxis=dict(tickangle=-45), font=dict(size=11))
fig_hm.show()

In [54]:
# === Cell 10: Combined Plotly dashboard (Isolated Figure Fix) ============
from plotly.subplots import make_subplots
import plotly.graph_objects as go

latest_p = float(now_prob.iloc[-1] * 100)

# 1. Create a standalone clean layout for the Gauge
fig_gauge = go.Figure(go.Indicator(
    mode='gauge+number', value=latest_p, number=dict(suffix='%'),
    title=dict(text="Recession Probability (nowcast)", font=dict(size=16)),
    gauge=dict(axis=dict(range=[0, 100]), bar=dict(color=PALETTE['line']),
               steps=[dict(range=[0, 33], color='#d9f0d3'),
                      dict(range=[33, 66], color='#fee08b'),
                      dict(range=[66, 100], color='#fdae61')],
               threshold=dict(line=dict(color='red', width=4), value=50))
))
fig_gauge.update_layout(height=350, margin=dict(t=50, b=20, l=20, r=20))

# 2. Create a clean 3-panel layout for all grid data plots
fig_grid = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Composite Activity Index', 'Top Indicator Contributions', '12m-Ahead Recession Probability'),
    specs=[[{'type': 'xy'}, {'type': 'xy'}],
           [{'type': 'xy', 'colspan': 2}, None]], # Span leading model across bottom
    vertical_spacing=0.18, horizontal_spacing=0.12
)

# Panel A: Composite Activity Index
fig_grid.add_trace(go.Scatter(x=composite.index, y=composite.values, mode='lines',
                              line=dict(color=PALETTE['line'], width=2), showlegend=False), row=1, col=1)
fig_grid.add_hline(y=0, line_dash='dash', line_color='gray', row=1, col=1)

# Panel B: Contributions
latest_z = z_filled.iloc[-1]
contrib = ((latest_z - pca_mean) * loadings) / pc1.std(ddof=0)
contrib = contrib.sort_values()
contrib.index = [NAMES[c] for c in contrib.index]
top = contrib.reindex(contrib.abs().sort_values().index).tail(12)
fig_grid.add_trace(go.Bar(x=top.values, y=top.index, orientation='h', showlegend=False,
    marker_color=[PALETTE['strong'] if v >= 0 else PALETTE['weak'] for v in top.values]), row=1, col=2)

# Panel C: Leading Probability
if HAS_SLOPE:
    fig_grid.add_trace(go.Scatter(x=lead_prob.index, y=lead_prob.values * 100, mode='lines',
                                  line=dict(color=PALETTE['lead'], width=2), showlegend=False), row=2, col=1)
    fig_grid.add_hline(y=50, line_dash='dash', line_color='gray', row=2, col=1)

# Apply backgrounds using the reliable helper function safely
add_recession_shading(fig_grid, row=1, col=1)
if HAS_SLOPE:
    add_recession_shading(fig_grid, row=2, col=1)

# Format axes labels
fig_grid.update_yaxes(title_text='std dev', row=1, col=1)
fig_grid.update_xaxes(title_text='Contribution Size', row=1, col=2)
fig_grid.update_yaxes(title_text='%', range=[0, 100], row=2, col=1)
fig_grid.update_layout(height=650, title_text='Macro Tracking & Leading Metrics', showlegend=False)

# Display both sequentially
fig_gauge.show()
fig_grid.show()

In [55]:
# Force-generate the output data metrics and text note files
try:
    z.to_csv('macro_zscores.csv')
    out = composite.to_frame('composite').join(now_prob.rename('nowcast_prob'))
    if 'T10Y3M' in features.columns:
        out = out.join(lead_prob.rename('lead_prob'))
    out.to_csv('macro_composite.csv')

    with open('macro_note.txt', 'w') as f:
        f.write("Macro tracking index and recession probability metrics verified successfully.")
    print("✅ CSV and TXT files generated successfully!")
except Exception as e:
    print("Error saving data files:", str(e))

✅ CSV and TXT files generated successfully!


In [56]:
%%writefile app.py
import os
import numpy as np
import pandas as pd
import streamlit as st
import plotly.graph_objects as go
from sklearn.decomposition import PCA
import statsmodels.api as sm
from fredapi import Fred

st.set_page_config(page_title='Macro Nowcasting Dashboard', layout='wide')
st.title('Macro Nowcasting Dashboard')
st.caption('Composite activity index and recession probability from FRED data.')

# Simple deployment placeholder app logic
st.info("Macro deployment script ready for production cloud integration.")

Writing app.py
